# 📓 Semana 3 · Dia 5 — Internals: partições, shuffle, cache e o Spark UI

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (performance) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Análise do Spark UI de um job real |

---


## 📖 Teoria — Partições e paralelismo

Cada arquivo/tabela é lido em **partições**. O paralelismo = número de tasks simultâneas ≈ número de partições ativas × cores. Poucas partições = subutilização; muitas = overhead.

`spark.sql.shuffle.partitions` (padrão 200) define o nº de partições após um shuffle (groupBy/join). Ajuste com bom senso — a Semana 8 aprofunda.


## 📖 Teoria — Shuffle — o custo do movimento

Quando um `groupBy` ou `join` precisa juntar linhas que estão em partições diferentes, o Spark **embaralha** os dados pela rede (shuffle): grava em disco, move, reordena. **Shuffle é o operador mais caro do Spark.**

**Broadcast join**: se uma tabela é pequena (< 10 MB por padrão), o Spark a copia para cada executor e faz join sem shuffle — muito mais rápido.

> 🎯 **Dica de prova (DEP)**: escolher entre broadcast join (tabela pequena) e sort-merge join (tabelas grandes, shuffle) é pergunta clássica. O broadcast é automático até o limite, mas pode ser forçado com hint.


## 📖 Teoria — cache() vs persist()

Ambos guardam o DataFrame em memória para **reutilização** (evita recalcular).
- `cache()` = `persist(MEMORY_AND_DISK)`.
- `persist(level)` permite escolher: DISK_ONLY, MEMORY_ONLY, etc.
- São **lazy**: só materializam na primeira ação.
- Use em dados **reutilizados várias vezes**; em dados de 1 uso, cache é desperdício.
- Para desfazer: `df.unpersist()`.


### 💻 Na prática — Plano físico e Spark UI

Gere um job e observe o plano físico.


In [ ]:
# Forçar um broadcast join para ver no plano físico
categorias = spark.createDataFrame([("85123A","Decoração"), ("71053","Cozinha")], ["StockCode", "cat"])
df = spark.table("workspace.bronze.vendas_bronze").limit(50_000)
j = df.join(categorias, "StockCode", "left")
print(j.explain("formatted"))  # veja o BroadcastExchange no plano
print("Se vir BroadcastExchange, o Spark fez broadcast (sem shuffle).")

In [ ]:
# Shuffle real: groupBy
g = spark.table("workspace.bronze.vendas_bronze").groupBy("Country").count()
print(g.explain("formatted"))  # veja Exchange (shuffle) no plano
g.collect()

In [ ]:
# cache: reutilizar sem recalcular
import time
df_ouro = spark.table("workspace.bronze.vendas_bronze").filter("Quantity > 5")
df_ouro.cache()
t0 = time.time(); df_ouro.count(); t1 = time.time()
print(f"1a contagem (carrega cache): {t1-t0:.2f}s")
t0 = time.time(); df_ouro.count(); t1 = time.time()
print(f"2a contagem (do cache): {t1-t0:.2f}s")
df_ouro.unpersist()

> 🎯 **Dica de prova**: O Spark UI (abas Query/Jobs) mostra: DAG, estágios, tasks, tempos e skew. Saber ler 'shuffle read/write' e 'task time vs duration' diferencia sênior de júnior em entrevistas.


## 🎯 Exercícios de fixação

**1.** Explique em 2 frases o que é shuffle e por que é caro.

**2.** Quando o Spark usa broadcast join automaticamente? Como forçar?

**3.** Diferencie cache() de persist(DISK_ONLY).

**4.** Rode um join e veja no explain se foi broadcast ou sort-merge.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Shuffle

Movimentação de linhas entre partições para satisfazer groupBy/join; envolve disco + rede, sendo o operador mais caro. Evitar shuffle é a 1ª regra de tuning.

**2.** Broadcast

Quando uma das tabelas é pequena (< spark.sql.autoBroadcastJoinThreshold, 10MB). Forçar: `df.join(cat.hint('broadcast'), 'key')`.

**3.** cache vs persist

cache() = MEMORY_AND_DISK. persist() aceita outros níveis (DISK_ONLY, MEMORY_ONLY...). Ambos são lazy e liberados com unpersist().

**4.** Explain

Procure `BroadcastExchange` (broadcast) vs `Exchange` + `SortMergeJoin` (shuffle).



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*